# prime-rl S1: Offline Import Test (RTX 6000 Pro)

Goal: confirm prime-rl + its dependency stack can be installed and imported on the
RTX 6000 Pro Kaggle notebook with internet OFF, using only the wheels collected
by `prime-rl-offline-prep` (mounted as a kernel_source).

Success criteria:
- `pip install --no-index --find-links <wheels>` succeeds
- `python -c 'import prime_rl, torch, vllm, transformers'` succeeds
- `torch.cuda.is_available()` is True and reports the RTX 6000 Pro

We do NOT run any training here. That's S2/S3.

## 1. Recon mounted prep notebook output

In [ ]:
import os, subprocess

PREP = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
subprocess.run(f"ls -la {PREP}", shell=True)
print()
# Output may be at PREP/ or PREP/output/. Detect.
if os.path.exists(f"{PREP}/output"):
    BASE = f"{PREP}/output"
elif os.path.exists(f"{PREP}/wheels"):
    BASE = PREP
else:
    raise RuntimeError(f"Cannot find wheels under {PREP}")
print(f"BASE = {BASE}")
subprocess.run(f"ls -la {BASE}", shell=True)
print()
WHEELS = f"{BASE}/wheels"
PRIME_RL_SRC = f"{BASE}/prime_rl_src"
subprocess.run(f"ls {WHEELS} | wc -l", shell=True)
subprocess.run(f"du -sh {WHEELS}", shell=True)

## 2. GPU sanity

In [ ]:
import subprocess
subprocess.run("nvidia-smi --query-gpu=name,memory.total,driver_version,compute_cap --format=csv", shell=True)
subprocess.run("python3.12 --version", shell=True)

## 3. Install prime-rl from offline wheels

`pip install --no-index --find-links <wheels> prime_rl` — pip resolves prime_rl from the
wheel directory and pulls every transitive dep from the same dir. No internet.

In [ ]:
import os, subprocess, glob, re

PREP = "/kaggle/input/notebooks/titanic12/prime-rl-offline-prep"
BASE = f"{PREP}/output" if os.path.exists(f"{PREP}/output") else PREP
WHEELS = f"{BASE}/wheels"

# Wheel set has duplicate torch versions, AND torch 2.11.0+cu128 needs NCCL 2.28
# but our wheels only have NCCL 2.27.5 (paired with torch 2.10). Kaggle's RTX 6000
# Pro image already ships torch 2.11.0+cu128 with the matching NCCL. To avoid the
# `undefined symbol: ncclDevCommCreate` ABI break, leave the GPU stack as the
# system installed it: skip every torch* wheel and every nvidia_*_cu12 wheel.
SKIP_PREFIXES = ("torch-", "torchvision-", "torchaudio-", "nvidia_")

def parse_version(name):
    m = re.match(r"([a-zA-Z0-9_]+?)-(\d+(?:\.\d+)*)", os.path.basename(name))
    if not m:
        return None, None
    return m.group(1).lower(), tuple(int(x) for x in m.group(2).split("."))

all_wheels = sorted(glob.glob(f"{WHEELS}/*.whl"))

# Step 1: drop GPU-stack wheels
gpu_dropped = [w for w in all_wheels if os.path.basename(w).startswith(SKIP_PREFIXES)]
remaining = [w for w in all_wheels if not os.path.basename(w).startswith(SKIP_PREFIXES)]

# Step 2: keep only highest version per package
keep = {}
dup_dropped = []
for w in remaining:
    pkg, ver = parse_version(w)
    if pkg is None:
        keep[os.path.basename(w)] = (None, w)
        continue
    if pkg in keep and keep[pkg][0] is not None:
        prev_ver, prev_path = keep[pkg]
        if ver > prev_ver:
            dup_dropped.append(prev_path)
            keep[pkg] = (ver, w)
        else:
            dup_dropped.append(w)
    else:
        keep[pkg] = (ver, w)

filtered = [v[1] for v in keep.values()]
print(f"total wheels: {len(all_wheels)}")
print(f"  GPU-stack skipped (system supplies): {len(gpu_dropped)}")
print(f"  duplicates dropped:                  {len(dup_dropped)}")
print(f"  -> installing:                       {len(filtered)}")

# Bypass pip's resolver entirely (vllm 0.19 pins transformers<5 but prime-rl
# patches vllm at import time via the transformers_v5_compat plugin).
cmd = (
    "python3.12 -m pip install --no-index --no-build-isolation --pre --no-deps "
    + " ".join(filtered)
)
subprocess.run(cmd, shell=True, check=True)

## 4. Import test

Try to import the core modules. We catch each import individually so a failure on
one doesn't hide the others.

In [ ]:
results = {}

for mod in ["torch", "transformers", "vllm", "prime_rl"]:
    try:
        m = __import__(mod)
        results[mod] = ("ok", getattr(m, "__version__", "?"))
    except Exception as e:
        results[mod] = ("FAIL", repr(e))

print("--- import results ---")
for mod, (status, info) in results.items():
    print(f"  {mod:15s} {status:5s} {info}")

import torch
print()
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"Compute capability: {torch.cuda.get_device_capability(0)}")

# Final verdict
if any(v[0] == "FAIL" for v in results.values()):
    raise RuntimeError("One or more imports failed -- see results above")
if not torch.cuda.is_available():
    raise RuntimeError("GPU not visible to torch")
print("\nS1 PASS")